***Support vector Machine***

In [902]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import time  # Importing time module to measure execution time

# Feature Extraction Functions
def compute_avg_pixel_intensity(image):
    """Compute the average pixel intensity of an image."""
    return np.mean(image)

def compute_symmetry_features(image):
    """Compute horizontal and vertical symmetry features."""
    vertical_symmetry = np.mean(image[:, :image.shape[1] // 2] - image[:, image.shape[1] // 2:])
    horizontal_symmetry = np.mean(image[:image.shape[0] // 2, :] - image[image.shape[0] // 2:, :])
    return vertical_symmetry, horizontal_symmetry

def compute_thickness(image):
    """Estimate the stroke thickness based on edge detection."""
    edges = cv2.Canny(image, 100, 200)
    return np.sum(edges) / (image.shape[0] * image.shape[1])

def compute_gaps_between_contours(image):
    """Calculate the average gap between detected contours."""
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) < 2:
        return 0
    centers = [cv2.boundingRect(c)[0] + cv2.boundingRect(c)[2] // 2 for c in contours]
    centers.sort()
    gaps = [centers[i + 1] - centers[i] for i in range(len(centers) - 1)]
    return np.mean(gaps) if gaps else 0

def compute_slant_using_hough(image):
    """Detect slant using the Hough Transform."""
    edges = cv2.Canny(image, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)
    if lines is not None:
        angles = [np.rad2deg(np.arctan2(np.sin(theta), np.cos(theta))) for rho, theta in lines[:, 0, :]]
        return np.mean(angles)
    return 0

def compute_endline_length(image):
    """Calculate the length of the endline (bottom-most stroke)."""
    edges = cv2.Canny(image, 100, 200)
    bottom_row = edges[-1, :]
    return np.sum(bottom_row) / 255

# Combine all features
def extract_features(image_path):
    """Extract all features for a given image."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Image at {image_path} could not be loaded.")
    
    # Preprocess: Resize image for consistency
    image = cv2.resize(image, (100, 100))  # Resize to 100x100 pixels
    
    # Extract features
    features = [
        compute_avg_pixel_intensity(image),
        *compute_symmetry_features(image),
        compute_thickness(image),
        compute_gaps_between_contours(image),
        compute_slant_using_hough(image),
        compute_endline_length(image)
    ]
    return np.array(features)

# Load Dataset
def load_dataset(dataset_path):
    """Load dataset and extract features and labels."""
    X, y = [], []
    for emotion in os.listdir(dataset_path):
        emotion_folder = os.path.join(dataset_path, emotion)
        if os.path.isdir(emotion_folder):
            for image_file in os.listdir(emotion_folder):
                image_path = os.path.join(emotion_folder, image_file)
                try:
                    features = extract_features(image_path)
                    X.append(features)
                    y.append(emotion)
                except Exception as e:
                    print(f"Error processing {image_path}: {e}")
    return np.array(X), np.array(y)

def train_and_evaluate_svm(X, y, test_size):
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Ensure that test_size is large enough to have samples from all classes
    n_classes = len(np.unique(y_encoded))
    if test_size < 1 / n_classes:
        print("Increasing test size.")
        test_size = 1 / n_classes

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=test_size, stratify=y_encoded)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=10, gamma='scale'))
    ])
    
    # Measure training time
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Measure prediction time
    start_time = time.time()
    y_pred = pipeline.predict(X_test)
    prediction_time = time.time() - start_time
    
    accuracy = accuracy_score(y_test, y_pred)

    # Store the accuracy with its corresponding test size and predicted emotions
    results = {
        'test_size': test_size,
        'accuracy': accuracy,
        'predicted_emotions': le.inverse_transform(y_pred),
        'training_time': training_time,
        'prediction_time': prediction_time
    }

    print(f"Accuracy: {accuracy}")
    print(f"Predicted Emotions: {results['predicted_emotions']}\n")
    print(f"Training Time: {training_time:.4f} seconds")
    print(f"Prediction Time: {prediction_time:.4f} seconds\n")
    
    return accuracy, pipeline, le, results


if __name__ == "__main__":
    dataset_path = r"C:\ML_DL_Files\dataset\Person_1"  # Replace with your dataset path
    test_image_path = r"C:\ML_DL_Files\Pictures\Krishna.jpg"  # Replace with your test image path

    print("Loading dataset...")
    X, y = load_dataset(dataset_path)

    best_accuracy = 0
    best_model, best_encoder = None, None
    all_results = []  # Store all accuracy results

    for test_size in [0.1, 0.2, 0.3]:
        print(f"Evaluating with test size {test_size * 100}%")
        accuracy, model, label_encoder, results = train_and_evaluate_svm(X, y, test_size)
        all_results.append(results)  # Append the results for each test_size

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model, best_encoder = model, label_encoder

    # Print all accuracy results with corresponding emotions
    print("\nAll Accuracy Results:")
    for result in all_results:
        print(f"Test Size: {result['test_size'] * 100}% - Accuracy: {result['accuracy']}")
        print(f"Predicted Emotions: {result['predicted_emotions']}")
        print(f"Training Time: {result['training_time']:.4f} seconds")
        print(f"Prediction Time: {result['prediction_time']:.4f} seconds\n")

    print("Best accuracy achieved:", best_accuracy)

    print("Predicting new image emotion...")
    predicted_emotion = predict_emotion(best_model, best_encoder, test_image_path)
    if predicted_emotion:
        print(f"Predicted Emotion: {predicted_emotion}")


Loading dataset...
Evaluating with test size 10.0%
Increasing test size.
Accuracy: 0.4
Predicted Emotions: ['Happy' 'Normal' 'Sad' 'Angry' 'Happy']

Training Time: 0.0030 seconds
Prediction Time: 0.0000 seconds

Evaluating with test size 20.0%
Increasing test size.
Accuracy: 0.4
Predicted Emotions: ['Normal' 'Happy' 'Normal' 'Normal' 'Angry']

Training Time: 0.0010 seconds
Prediction Time: 0.0010 seconds

Evaluating with test size 30.0%
Accuracy: 0.8333333333333334
Predicted Emotions: ['Angry' 'Sad' 'Happy' 'Happy' 'Sad' 'Angry']

Training Time: 0.0020 seconds
Prediction Time: 0.0000 seconds


All Accuracy Results:
Test Size: 25.0% - Accuracy: 0.4
Predicted Emotions: ['Happy' 'Normal' 'Sad' 'Angry' 'Happy']
Training Time: 0.0030 seconds
Prediction Time: 0.0000 seconds

Test Size: 25.0% - Accuracy: 0.4
Predicted Emotions: ['Normal' 'Happy' 'Normal' 'Normal' 'Angry']
Training Time: 0.0010 seconds
Prediction Time: 0.0010 seconds

Test Size: 30.0% - Accuracy: 0.8333333333333334
Predicted E

In [511]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Function for cross-validation with custom train-test splits
def cross_validate_svm(X, y, test_size):
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Ensure there are enough samples in the test set to represent each class
    n_classes = len(set(y_encoded))
    min_test_size = max(test_size, 1 / n_classes)  # Ensure at least 1 sample per class

    # Perform the specified train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=min_test_size, stratify=y_encoded)

    # Create a pipeline with scaling and SVM
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=10, gamma='scale'))
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Return the accuracy
    return accuracy

if __name__ == "__main__":
    dataset_path = r"C:\ML_DL_Files\dataset\Person_1"  # Replace with your dataset path
    test_image_path = r"C:\ML_DL_Files\Pictures\Krishna.jpg"  # Replace with your test image path

    print("Loading dataset...")
    X, y = load_dataset(dataset_path)

    # Test splits to evaluate
    test_sizes = [0.1, 0.2, 0.3]

    # Loop through the test sizes
    for test_size in test_sizes:
        print(f"\nEvaluating with test size {test_size * 100}%:")
        accuracy = cross_validate_svm(X, y, test_size)
        print(f"Accuracy: {accuracy}")


Loading dataset...

Evaluating with test size 10.0%:
Accuracy: 0.4

Evaluating with test size 20.0%:
Accuracy: 0.8

Evaluating with test size 30.0%:
Accuracy: 0.5


***Random Forest Classifier***

In [915]:
import os
import cv2
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# Feature Extraction Functions
def compute_avg_pixel_intensity(image):
    """Compute the average pixel intensity of an image."""
    return np.mean(image)

def compute_symmetry_features(image):
    """Compute horizontal and vertical symmetry features."""
    vertical_symmetry = np.mean(image[:, :image.shape[1] // 2] - image[:, image.shape[1] // 2:])
    horizontal_symmetry = np.mean(image[:image.shape[0] // 2, :] - image[image.shape[0] // 2:, :])
    return vertical_symmetry, horizontal_symmetry

def compute_thickness(image):
    """Estimate the stroke thickness based on edge detection."""
    edges = cv2.Canny(image, 100, 200)
    return np.sum(edges) / (image.shape[0] * image.shape[1])

def compute_gaps_between_contours(image):
    """Calculate the average gap between detected contours."""
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) < 2:
        return 0
    centers = [cv2.boundingRect(c)[0] + cv2.boundingRect(c)[2] // 2 for c in contours]
    centers.sort()
    gaps = [centers[i + 1] - centers[i] for i in range(len(centers) - 1)]
    return np.mean(gaps) if gaps else 0

def compute_slant_using_hough(image):
    """Detect slant using the Hough Transform."""
    edges = cv2.Canny(image, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)
    if lines is not None:
        angles = [np.rad2deg(np.arctan2(np.sin(theta), np.cos(theta))) for rho, theta in lines[:, 0, :]]
        return np.mean(angles)
    return 0

def compute_endline_length(image):
    """Calculate the length of the endline (bottom-most stroke)."""
    edges = cv2.Canny(image, 100, 200)
    bottom_row = edges[-1, :]
    return np.sum(bottom_row) / 255

# Combine all features
def extract_features(image_path):
    """Extract all features for a given image."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Image at {image_path} could not be loaded.")
    
    # Preprocess: Resize image for consistency
    image = cv2.resize(image, (100, 100))  # Resize to 100x100 pixels
    
    # Extract features
    features = [
        compute_avg_pixel_intensity(image),
        *compute_symmetry_features(image),
        compute_thickness(image),
        compute_gaps_between_contours(image),
        compute_slant_using_hough(image),
        compute_endline_length(image)
    ]
    return np.array(features)

# Load Dataset
def load_dataset(dataset_path):
    """Load dataset and extract features and labels."""
    X, y = [], []
    for emotion in os.listdir(dataset_path):
        emotion_folder = os.path.join(dataset_path, emotion)
        if os.path.isdir(emotion_folder):
            for image_file in os.listdir(emotion_folder):
                image_path = os.path.join(emotion_folder, image_file)
                try:
                    features = extract_features(image_path)
                    X.append(features)
                    y.append(emotion)
                except Exception as e:
                    print(f"Error processing {image_path}: {e}")
    return np.array(X), np.array(y)

def train_and_evaluate_random_forest(X, y, test_size):
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Ensure that test_size is large enough to have samples from all classes
    n_classes = len(np.unique(y_encoded))
    if test_size < 1 / n_classes:
        print("Increasing test size.")
        test_size = 1 / n_classes

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=test_size, stratify=y_encoded)

    # Create a pipeline with scaling and RandomForest
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestClassifier(n_estimators=100))
    ])
    
    # Start time for training
    start_time = time.time()
    
    pipeline.fit(X_train, y_train)

    # End time for training
    training_time = time.time() - start_time

    # Start time for prediction
    start_prediction_time = time.time()
    
    y_pred = pipeline.predict(X_test)
    
    # End time for prediction
    prediction_time = time.time() - start_prediction_time

    accuracy = accuracy_score(y_test, y_pred)

    # Store the accuracy with its corresponding test size and predicted emotions
    results = {
        'test_size': test_size,
        'accuracy': accuracy,
        'predicted_emotions': le.inverse_transform(y_pred),
        'training_time': training_time,
        'prediction_time': prediction_time
    }

    print(f"Accuracy: {accuracy}")
    print(f"Predicted Emotions: {results['predicted_emotions']}\n")
    print(f"Training Time: {training_time} seconds")
    print(f"Prediction Time: {prediction_time} seconds\n")
    
    return accuracy, pipeline, le, results


if __name__ == "__main__":
    dataset_path = r"C:\ML_DL_Files\dataset\Person_1"  # Replace with your dataset path
    test_image_path = r"C:\ML_DL_Files\Pictures\Krishna.jpg"  # Replace with your test image path

    print("------------------Loading dataset---------------")
    print()
    start_loading_time = time.time()
    X, y = load_dataset(dataset_path)
    loading_time = time.time() - start_loading_time
    print(f"Dataset loaded in {loading_time} seconds")

    best_accuracy = 0
    best_model, best_encoder = None, None
    all_results = []  # Store all accuracy results

    for test_size in [0.1, 0.2, 0.3]:
        print(f"Evaluating with test size {test_size * 100}%")
        accuracy, model, label_encoder, results = train_and_evaluate_random_forest(X, y, test_size)
        all_results.append(results)  # Append the results for each test_size
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model, best_encoder = model, label_encoder
    
    # Print all accuracy results with corresponding emotions
    print("\nAll Accuracy Results:")
    for result in all_results:
        print(f"Test Size: {result['test_size'] * 100}% - Accuracy: {result['accuracy']}")
        print(f"Predicted Emotions: {result['predicted_emotions']}\n")
        
    print("Best accuracy achieved:", best_accuracy)

    # Optionally, to predict for a new image:
    print("Predicting new image emotion...")
    predicted_emotion = predict_emotion(best_model, best_encoder, test_image_path)
    if predicted_emotion:
        print(f"Predicted Emotion: {predicted_emotion}")


------------------Loading dataset---------------

Dataset loaded in 0.12219071388244629 seconds
Evaluating with test size 10.0%
Increasing test size.
Accuracy: 0.4
Predicted Emotions: ['Happy' 'Angry' 'Sad' 'Happy' 'Normal']

Training Time: 0.24851083755493164 seconds
Prediction Time: 0.0 seconds

Evaluating with test size 20.0%
Increasing test size.
Accuracy: 0.8
Predicted Emotions: ['Sad' 'Sad' 'Normal' 'Sad' 'Angry']

Training Time: 0.1570420265197754 seconds
Prediction Time: 0.0 seconds

Evaluating with test size 30.0%
Accuracy: 0.3333333333333333
Predicted Emotions: ['Angry' 'Happy' 'Happy' 'Normal' 'Normal' 'Happy']

Training Time: 0.1646406650543213 seconds
Prediction Time: 0.013010978698730469 seconds


All Accuracy Results:
Test Size: 25.0% - Accuracy: 0.4
Predicted Emotions: ['Happy' 'Angry' 'Sad' 'Happy' 'Normal']

Test Size: 25.0% - Accuracy: 0.8
Predicted Emotions: ['Sad' 'Sad' 'Normal' 'Sad' 'Angry']

Test Size: 30.0% - Accuracy: 0.3333333333333333
Predicted Emotions: ['A

In [1058]:
if __name__ == "__main__":
    dataset_path = r"C:\ML_DL_Files\dataset\Person_1"  # Replace with your dataset path

    print("------------------Loading dataset---------------")
    print()
    start_loading_time = time.time()
    X, y = load_dataset(dataset_path)
    loading_time = time.time() - start_loading_time
    print(f"Dataset loaded in {loading_time} seconds")

    best_accuracy_rf = 0
    best_accuracy_svm = 0
    best_model_rf, best_model_svm = None, None
    best_encoder_rf, best_encoder_svm = None, None
    all_results_rf = []  # Store all accuracy results for Random Forest
    all_results_svm = []  # Store all accuracy results for SVM

    # Evaluate Random Forest
    for test_size in [0.1, 0.2, 0.3]:
        print(f"Evaluating Random Forest with test size {test_size * 100}%")
        accuracy_rf, model_rf, label_encoder_rf, results_rf = train_and_evaluate_random_forest(X, y, test_size)
        all_results_rf.append(results_rf)  # Append the results for Random Forest
        if accuracy_rf > best_accuracy_rf:
            best_accuracy_rf = accuracy_rf
            best_model_rf, best_encoder_rf = model_rf, label_encoder_rf

    # Evaluate SVM
    for test_size in [0.1, 0.2, 0.3]:
        print(f"Evaluating SVM with test size {test_size * 100}%")
        accuracy_svm, model_svm, label_encoder_svm, results_svm = train_and_evaluate_svm(X, y, test_size)
        all_results_svm.append(results_svm)  # Append the results for SVM
        if accuracy_svm > best_accuracy_svm:
            best_accuracy_svm = accuracy_svm
            best_model_svm, best_encoder_svm = model_svm, label_encoder_svm
    
    # Print all accuracy results for Random Forest and SVM
    print("\nAll Accuracy Results for Random Forest:")
    for result_rf in all_results_rf:
        print(f"Test Size: {result_rf['test_size'] * 100}% - Accuracy: {result_rf['accuracy']}")
        print(f"Predicted Emotions: {result_rf['predicted_emotions']}\n")
    
    print("\nAll Accuracy Results for SVM:")
    for result_svm in all_results_svm:
        print(f"Test Size: {result_svm['test_size'] * 100}% - Accuracy: {result_svm['accuracy']}")
        print(f"Predicted Emotions: {result_svm['predicted_emotions']}\n")

    print("Best accuracy achieved by Random Forest:", best_accuracy_rf)
    print("Best accuracy achieved by SVM:", best_accuracy_svm)

    print("\nModel Accuracy Comparison")
    print("------------------------------------------------")
    print("Models                 | ACCURACY ")
    print("------------------------------------------------")
    print(f"Random Forest         | {best_accuracy_rf}") 
    print(f"SVM                   | {best_accuracy_svm}")
    # Include other models like ANN here if you have them
    print("------------------------------------------------")


------------------Loading dataset---------------

Dataset loaded in 0.08154106140136719 seconds
Evaluating Random Forest with test size 10.0%
Increasing test size.
Accuracy: 0.6
Predicted Emotions: ['Angry' 'Sad' 'Happy' 'Happy' 'Happy']

Training Time: 0.1732006072998047 seconds
Prediction Time: 0.005982637405395508 seconds

Evaluating Random Forest with test size 20.0%
Increasing test size.
Accuracy: 0.8
Predicted Emotions: ['Angry' 'Sad' 'Sad' 'Sad' 'Normal']

Training Time: 0.14963793754577637 seconds
Prediction Time: 0.0 seconds

Evaluating Random Forest with test size 30.0%
Accuracy: 0.3333333333333333
Predicted Emotions: ['Happy' 'Sad' 'Happy' 'Happy' 'Angry' 'Sad']

Training Time: 0.15160775184631348 seconds
Prediction Time: 0.015605449676513672 seconds

Evaluating SVM with test size 10.0%
Increasing test size.
Accuracy: 0.4
Predicted Emotions: ['Normal' 'Angry' 'Sad' 'Normal' 'Sad']

Training Time: 0.0 seconds
Prediction Time: 0.0 seconds

Evaluating SVM with test size 20.0%
I